In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import deepchem as dc
#from pubchempy import get_cids, get_compounds

#import tensorflow as tf
#physical_devices = tf.config.list_physical_devices('GPU')
#tf.config.experimental.set_memory_growth(physical_devices[0], enable=True)

import matplotlib.pyplot as plt
#import seaborn as sns

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler

In [ ]:
def display_df(df):
    display(HTML(df.to_html()))
    return None

In [ ]:
class CollinearColumnRemover(BaseEstimator, TransformerMixin):
    def __init__(self, threshold, col_regex=None, exclude_cols=None):
        """
        :param threshold: float in [0, 1], if two columns have correlation greater than threshold
                          one of them will be removed
        :param col_regex: str, regular expression to select columns
        """
        self._threshold = threshold
        self._col_regex = col_regex
        if exclude_cols is None:
            self._exclude_cols = []
        else:
            self._exclude_cols = exclude_cols
    
    def _collinear_columns(self, df, threshold):
        if self._col_regex is None:
            df_sel = df.select_dtypes(["number", "bool"])
        else:
            df_sel = df.filter(regex=self._col_regex)
            df_sel = df_sel.select_dtypes(["number", "bool"])
        
        df_sel = df_sel.astype("float32")
        
        all_cols = df_sel.columns.to_list()
        all_cols = [col for col in all_cols if col not in self._exclude_cols]
        df_sel = df_sel[all_cols]
        ncols = len(all_cols)
        
        corr_mat = df_sel.corr().abs()
        self._corr_mat = corr_mat
        collin_cols = []
        for i in range(ncols-1):
            col_i = all_cols[i]
            if col_i in collin_cols:
                continue
            
            for j in range(i + 1, ncols):
                col_j = all_cols[j]
                if col_j in collin_cols:
                    continue
                
                corr = corr_mat.loc[col_i, col_j]
                if corr > threshold:
                    collin_cols.append(col_j)
        
        collin_cols = list(set(collin_cols))
        return collin_cols
    
    
    def fit(self, df):
        self._collin_cols = self._collinear_columns(df, self._threshold)
        return self
    
    def transform(self, df):
        all_cols = df.columns.to_list()
        nonexist_cols = [col for col in self._collin_cols if col not in all_cols]
        if len(nonexist_cols) > 0:
            print("WARNING: These collinear cols to be droped do not exist in df:", nonexist_cols)
            
        droped_col = [col for col in self._collin_cols if col in all_cols]
        print("Number of columns droped due to collinearity:", len(droped_col))
        return df.drop(droped_col, axis="columns")

In [ ]:
class NumImputer(BaseEstimator, TransformerMixin):
    def __init__(self, method="mean", exclude_cols=None):
        self._method = method
        if exclude_cols is None:
            self._exclude_cols = []
        else:
            self._exclude_cols = exclude_cols
    
    def fit(self, df_train):
        num_cols = df_train.select_dtypes(["number"]).columns.to_list()
        num_cols = [col for col in num_cols if col not in self._exclude_cols]
        
        self._train_cols = df_train.columns.to_list()
        
        self._impute_values = {}
        for col in num_cols:
            self._impute_values[col] = df_train[col].agg(self._method)
        return self
    
    def transform(self, df):
        df = df.copy()
        cols = df.columns.to_list()
        assert set(cols) == set(self._train_cols), "Do not have the same set of cols as train"
        
        for col, val in self._impute_values.items():
            if df[col].isnull().sum() > 0:
                df[col] = df[col].fillna(val)
        
        # align columns
        df = df[self._train_cols]
        return df
    

class CatImputer(BaseEstimator, TransformerMixin):
    def __init__(self, val="MISSING"):
        self._val = val
    
    def fit(self, df_train):
        cat_cols = df_train.select_dtypes(["object", "category", "bool"]).columns.to_list()
        self._train_cols = df_train.columns.to_list()
        
        self._impute_values = {}
        for col in cat_cols:
            self._impute_values[col] = self._val
        return self
    
    def transform(self, df):
        df = df.copy()
        cols = df.columns.to_list()
        assert set(cols) == set(self._train_cols), "Do not have the same set of cols as train"
        
        for col, val in self._impute_values.items():
            if df[col].isnull().sum() > 0:
                df[col] = df[col].astype("object").fillna(val).astype("category")
                
        # align columns
        df = df[self._train_cols]
        return df

In [ ]:
class Standardizer(BaseEstimator, TransformerMixin):
    def __init__(self, exclude_cols=None, to_array=False):
        if exclude_cols is None:
            self._exclude_cols = []
        else:
            self._exclude_cols = exclude_cols
            
        self._to_array = to_array
        
    def fit(self, df_train):
        num_cols = df_train.select_dtypes(["number"]).columns.to_list()
        num_cols = [col for col in num_cols if col not in self._exclude_cols]
        
        self._mean = {col: df_train[col].mean() for col in num_cols}
        self._std = {col: df_train[col].std() for col in num_cols}
        return self
    
    def transform(self, df):
        for col in self._mean:
            if self._std[col] > 0:
                df[col] = (df[col] - self._mean[col]) / self._std[col]
                df[col] = df[col].astype("float32")
            else:
                print("WARNING: " + col + " has zero std.")
                df[col] = df[col] - self._mean[col]
                df[col] = df[col].astype("float32")
                
        if self._to_array:
            return df.values.astype(np.float32)
        else:
            return df

In [ ]:
def check_smiles_in(simles, smiles_lst):
    assert isinstance(simles, str)
    assert isinstance(smiles_lst, list)
    return simles.lower().strip() in [s.lower().strip() for s in smiles_lst]

# pdY

## train/test

In [ ]:
assert False
df00 = pd.read_csv("data/process/clean_train_test_800.csv")
print(df00.shape)
df00["new_id"] = ["_" + "{}".format(i) for i in range(df00.shape[0])] 
df00["code"] = "labeled"
df00["smiles_len"] = df00["smiles"].apply(len)
cnames = ["new_id", "smiles", "code", "smiles_len", "dG", "train_test"]
df00 = df00[cnames]
print(df00.shape)
df00.to_csv("data/process/pdY_ab42_train_test.csv", index=False)
df00.head()

In [ ]:
pd.read_csv("data/process/pdY_chembl33.csv").head(1)

## chembl36excl33

In [ ]:
assert False

df00 = pd.read_csv("data/raw/chembl_36excl33.csv")
print(df00.shape)
df00["new_id"] = df00["chembl_id"] 
df00["code"] = "chembl36excl33"
df00["smiles_len"] = df00["smiles"].apply(len)
df00["dG"] = np.nan
df00["train_test"] = "pred"
cnames = ["new_id", "smiles", "code", "smiles_len", "dG", "train_test"]
df00 = df00[cnames]
print(df00.shape)
df00.to_csv("data/process/pdY_chembl36excl33.csv", index=False)
df00.head()

## amylovis201

In [ ]:
assert False
smiles = ["c1ccc2c(c1)c(ccc2)NC(=O)CCC(=O)NCCNC(=S)SC"]
df00 = pd.DataFrame({"smiles": smiles})
df00["new_id"] = ["_{}".format(i) for i in range(df00.shape[0])]
df00["code"] = "amylovis201"
df00["smiles_len"] = df00["smiles"].apply(len)
df00["dG"] = np.nan
df00["train_test"] = "pred"
cnames = ["new_id", "smiles", "code", "smiles_len", "dG", "train_test"]
df00 = df00[cnames]
print(df00.shape)
df00.to_csv("data/process/pdY_amylovis201.csv", index=False)
df00.head()

## ref01

In [ ]:
assert False
smiles = ["CC1=C(C(C[C@@H](C1=O)O)(C)C)/C=C/C(=C/C=C/C(=C/C=C/C=C(/C=C/C=C(/C=C/C2=C(C(=O)[C@H](CC2(C)C)O)C)\C)\C)/C)/C",
          "COC1=C(C=C2C(=C1)CC(C2=O)CC3CCN(CC3)CC4=CC=CC=C4)OC",
          "C[C@]1(CS(=O)(=O)N(C(=N1)N)C)C2=C(C=CC(=C2)NC(=O)C3=NC=C(C=C3)F)F",
          "C1[C@H]2CSC(=N[C@]2(CN1C3=NC=CC=N3)C4=CC=CS4)N",
         ]

new_id = ["astaxanthin",
          "donepezil",
          "verubecestat",
          "compound6"
         ]
df00 = pd.DataFrame({"new_id": new_id, "smiles": smiles})
df00["code"] = "ref01"
df00["smiles_len"] = df00["smiles"].apply(len)
df00["dG"] = np.nan
df00["train_test"] = "pred"
cnames = ["new_id", "smiles", "code", "smiles_len", "dG", "train_test"]
df00 = df00[cnames]
print(df00.shape)
df00.to_csv("data/process/pdY_ref01.csv", index=False)
df00.head()

# pdX

## Extract RDKitDescriptors

### train_test

In [ ]:
assert False

pdY = pd.read_csv("data/process/pdY_ab42_train_test.csv")
print("pdY", pdY.shape)
display_df(pdY.head())

rdkit_featurizer = dc.feat.RDKitDescriptors()
X = rdkit_featurizer(pdY["smiles"])

X1 = []
for y in X:
    if y.shape[0] > 0:
        X1.append(y.tolist())
    else:
        y = [np.nan]*200
        X1.append(y)
X1 = np.array(X1)

X2 = pd.DataFrame(X1, columns=rdkit_featurizer.descriptors)
X2["new_id"] = pdY["new_id"]
X2["smiles"] = pdY["smiles"]
X2["dG"] = pdY["dG"]
X2["code"] = pdY["code"]
X2["train_test"] = pdY["train_test"]
X2["smiles_len"] = pdY["smiles_len"]
display_df(X2.head())

X2.to_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_200ft.csv", index=False)

## chembl33

In [ ]:
#!cp /data/hai/Compound_Screen/EGFR/data/process/pdXY_chembl33_rdkit_descriptors_200ft.csv data/process/.

In [ ]:
display_df(pd.read_csv("data/process/pdXY_chembl33_rdkit_descriptors_200ft.csv").head(1))

## chembl36excl33

In [ ]:
assert False

pdY = pd.read_csv("data/process/pdY_chembl36excl33.csv")
print("pdY", pdY.shape)
display_df(pdY.head())

rdkit_featurizer = dc.feat.RDKitDescriptors()
X = rdkit_featurizer(pdY["smiles"])

X1 = []
for y in X:
    if y.shape[0] > 0:
        X1.append(y.tolist())
    else:
        y = [np.nan]*200
        X1.append(y)
X1 = np.array(X1)

X2 = pd.DataFrame(X1, columns=rdkit_featurizer.descriptors)
X2["new_id"] = pdY["new_id"]
X2["smiles"] = pdY["smiles"]
X2["dG"] = pdY["dG"]
X2["code"] = pdY["code"]
X2["train_test"] = pdY["train_test"]
X2["smiles_len"] = pdY["smiles_len"]
display_df(X2.head())

X2.to_csv("data/process/pdXY_chembl36excl33_rdkit_descriptors_200ft.csv", index=False)

## amylovis201

In [ ]:
assert False

pdY = pd.read_csv("data/process/pdY_amylovis201.csv")
print("pdY", pdY.shape)
display_df(pdY.head())

rdkit_featurizer = dc.feat.RDKitDescriptors()
X = rdkit_featurizer(pdY["smiles"])

X1 = []
for y in X:
    if y.shape[0] > 0:
        X1.append(y.tolist())
    else:
        y = [np.nan]*200
        X1.append(y)
X1 = np.array(X1)

X2 = pd.DataFrame(X1, columns=rdkit_featurizer.descriptors)
X2["new_id"] = pdY["new_id"]
X2["smiles"] = pdY["smiles"]
X2["dG"] = pdY["dG"]
X2["code"] = pdY["code"]
X2["train_test"] = pdY["train_test"]
X2["smiles_len"] = pdY["smiles_len"]
display_df(X2.head())

X2.to_csv("data/process/pdXY_amylovis201_rdkit_descriptors_200ft.csv", index=False)

## ref01

In [ ]:
assert False

pdY = pd.read_csv("data/process/pdY_ref01.csv")
print("pdY", pdY.shape)
display_df(pdY.head())

rdkit_featurizer = dc.feat.RDKitDescriptors()
X = rdkit_featurizer(pdY["smiles"])

X1 = []
for y in X:
    if y.shape[0] > 0:
        X1.append(y.tolist())
    else:
        y = [np.nan]*200
        X1.append(y)
X1 = np.array(X1)

X2 = pd.DataFrame(X1, columns=rdkit_featurizer.descriptors)
X2["new_id"] = pdY["new_id"]
X2["smiles"] = pdY["smiles"]
X2["dG"] = pdY["dG"]
X2["code"] = pdY["code"]
X2["train_test"] = pdY["train_test"]
X2["smiles_len"] = pdY["smiles_len"]
display_df(X2.head())

X2.to_csv("data/process/pdXY_ref01_rdkit_descriptors_200ft.csv", index=False)

## Remove mostly zero columns

### train_test

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_200ft.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()

mostly_zero_cols = []
for col in PDX_COLS:
    zero_rate = (pdXY_train[col] == 0).mean()
    if zero_rate > 0.95:
        print("{}    {}".format(col, zero_rate))
        mostly_zero_cols.append(col)

print("mostly_zero_cols", len(mostly_zero_cols))
print("there remain {}".format(len(PDX_COLS) - len(mostly_zero_cols)))

pdXY = pdXY.drop(mostly_zero_cols, axis=1)
print(pdXY.shape)
display_df(pdXY.head())

pdXY.to_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_136ft.csv", index=False)

## chembl33

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_200ft.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()

mostly_zero_cols = []
for col in PDX_COLS:
    zero_rate = (pdXY_train[col] == 0).mean()
    if zero_rate > 0.95:
        print("{}    {}".format(col, zero_rate))
        mostly_zero_cols.append(col)

print("mostly_zero_cols", len(mostly_zero_cols))
print("there remain {}".format(len(PDX_COLS) - len(mostly_zero_cols)))

df00 = pd.read_csv("data/process/pdXY_chembl33_rdkit_descriptors_200ft.csv")
df00 = df00.drop(mostly_zero_cols, axis=1)
print(df00.shape)
display_df(df00.head())

df00.to_csv("data/process/pdXY_chembl33_rdkit_descriptors_136ft.csv", index=False)

## chembl36excl33

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_200ft.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()

mostly_zero_cols = []
for col in PDX_COLS:
    zero_rate = (pdXY_train[col] == 0).mean()
    if zero_rate > 0.95:
        print("{}    {}".format(col, zero_rate))
        mostly_zero_cols.append(col)

print("mostly_zero_cols", len(mostly_zero_cols))
print("there remain {}".format(len(PDX_COLS) - len(mostly_zero_cols)))

df00 = pd.read_csv("data/process/pdXY_chembl36excl33_rdkit_descriptors_200ft.csv")
df00 = df00.drop(mostly_zero_cols, axis=1)
print(df00.shape)
display_df(df00.head())

df00.to_csv("data/process/pdXY_chembl36excl33_rdkit_descriptors_136ft.csv", index=False)

## amylovis201

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_200ft.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()

mostly_zero_cols = []
for col in PDX_COLS:
    zero_rate = (pdXY_train[col] == 0).mean()
    if zero_rate > 0.95:
        print("{}    {}".format(col, zero_rate))
        mostly_zero_cols.append(col)

print("mostly_zero_cols", len(mostly_zero_cols))
print("there remain {}".format(len(PDX_COLS) - len(mostly_zero_cols)))

df00 = pd.read_csv("data/process/pdXY_amylovis201_rdkit_descriptors_200ft.csv")
df00 = df00.drop(mostly_zero_cols, axis=1)
print(df00.shape)
display_df(df00.head())

df00.to_csv("data/process/pdXY_amylovis201_rdkit_descriptors_136ft.csv", index=False)

## ref01

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_200ft.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()

mostly_zero_cols = []
for col in PDX_COLS:
    zero_rate = (pdXY_train[col] == 0).mean()
    if zero_rate > 0.95:
        print("{}    {}".format(col, zero_rate))
        mostly_zero_cols.append(col)

print("mostly_zero_cols", len(mostly_zero_cols))
print("there remain {}".format(len(PDX_COLS) - len(mostly_zero_cols)))

df00 = pd.read_csv("data/process/pdXY_ref01_rdkit_descriptors_200ft.csv")
df00 = df00.drop(mostly_zero_cols, axis=1)
print(df00.shape)
display_df(df00.head())

df00.to_csv("data/process/pdXY_ref01_rdkit_descriptors_136ft.csv", index=False)

## Remove correlated columns

### train_test

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_136ft.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]
print("PDX_COLS", len(PDX_COLS))

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()
print("pdXY_train", pdXY_train.shape)

remover = CollinearColumnRemover(0.95, exclude_cols=PDY_COLS)
remover.fit(pdXY_train)

pdXY = remover.transform(pdXY)
print(pdXY.shape)
display_df(pdXY.head())


pdXY.to_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft.csv", index=False)

## chembl33

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_136ft.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]
print("PDX_COLS", len(PDX_COLS))

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()
print("pdXY_train", pdXY_train.shape)

remover = CollinearColumnRemover(0.95, exclude_cols=PDY_COLS)
remover.fit(pdXY_train)

df00 = pd.read_csv("data/process/pdXY_chembl33_rdkit_descriptors_136ft.csv")
df00 = remover.transform(df00)
print(df00.shape)
display_df(df00.head())


df00.to_csv("data/process/pdXY_chembl33_rdkit_descriptors_109ft.csv", index=False)

## chembl36excl33

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_136ft.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]
print("PDX_COLS", len(PDX_COLS))

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()
print("pdXY_train", pdXY_train.shape)

remover = CollinearColumnRemover(0.95, exclude_cols=PDY_COLS)
remover.fit(pdXY_train)

df00 = pd.read_csv("data/process/pdXY_chembl36excl33_rdkit_descriptors_136ft.csv")
df00 = remover.transform(df00)
print(df00.shape)
display_df(df00.head())


df00.to_csv("data/process/pdXY_chembl36excl33_rdkit_descriptors_109ft.csv", index=False)

## amylovis201

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_136ft.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]
print("PDX_COLS", len(PDX_COLS))

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()
print("pdXY_train", pdXY_train.shape)

remover = CollinearColumnRemover(0.95, exclude_cols=PDY_COLS)
remover.fit(pdXY_train)

df00 = pd.read_csv("data/process/pdXY_amylovis201_rdkit_descriptors_136ft.csv")
df00 = remover.transform(df00)
print(df00.shape)
display_df(df00.head())


df00.to_csv("data/process/pdXY_amylovis201_rdkit_descriptors_109ft.csv", index=False)

## ref01

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_136ft.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]
print("PDX_COLS", len(PDX_COLS))

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()
print("pdXY_train", pdXY_train.shape)

remover = CollinearColumnRemover(0.95, exclude_cols=PDY_COLS)
remover.fit(pdXY_train)

df00 = pd.read_csv("data/process/pdXY_ref01_rdkit_descriptors_136ft.csv")
df00 = remover.transform(df00)
print(df00.shape)
display_df(df00.head())


df00.to_csv("data/process/pdXY_ref01_rdkit_descriptors_109ft.csv", index=False)

## Impute missing values

### train_test

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]
print("PDX_COLS", len(PDX_COLS))

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()
print("pdXY_train", pdXY_train.shape)

imputer = NumImputer(method="median", exclude_cols=PDY_COLS)
imputer.fit(pdXY_train)

pdXY = imputer.transform(pdXY)
print(pdXY.shape)
display_df(pdXY.head())

pdXY.to_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft_imputed.csv", index=False)

## chembl33

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]
print("PDX_COLS", len(PDX_COLS))

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()
print("pdXY_train", pdXY_train.shape)

imputer = NumImputer(method="median", exclude_cols=PDY_COLS)
imputer.fit(pdXY_train)


df00 = pd.read_csv("data/process/pdXY_chembl33_rdkit_descriptors_109ft.csv")
df00 = imputer.transform(df00)
print(df00.shape)
display_df(df00.head())

df00.to_csv("data/process/pdXY_chembl33_rdkit_descriptors_109ft_imputed.csv", index=False)

## chembl36excl33

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]
print("PDX_COLS", len(PDX_COLS))

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()
print("pdXY_train", pdXY_train.shape)

imputer = NumImputer(method="median", exclude_cols=PDY_COLS)
imputer.fit(pdXY_train)


df00 = pd.read_csv("data/process/pdXY_chembl36excl33_rdkit_descriptors_109ft.csv")
df00 = imputer.transform(df00)
print(df00.shape)
display_df(df00.head())

df00.to_csv("data/process/pdXY_chembl36excl33_rdkit_descriptors_109ft_imputed.csv", index=False)

## amylovis201

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]
print("PDX_COLS", len(PDX_COLS))

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()
print("pdXY_train", pdXY_train.shape)

imputer = NumImputer(method="median", exclude_cols=PDY_COLS)
imputer.fit(pdXY_train)


df00 = pd.read_csv("data/process/pdXY_amylovis201_rdkit_descriptors_109ft.csv")
df00 = imputer.transform(df00)
print(df00.shape)
display_df(df00.head())

df00.to_csv("data/process/pdXY_amylovis201_rdkit_descriptors_109ft_imputed.csv", index=False)

## ref01

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]
print("PDX_COLS", len(PDX_COLS))

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()
print("pdXY_train", pdXY_train.shape)

imputer = NumImputer(method="median", exclude_cols=PDY_COLS)
imputer.fit(pdXY_train)


df00 = pd.read_csv("data/process/pdXY_ref01_rdkit_descriptors_109ft.csv")
df00 = imputer.transform(df00)
print(df00.shape)
display_df(df00.head())

df00.to_csv("data/process/pdXY_ref01_rdkit_descriptors_109ft_imputed.csv", index=False)

## Standardize

### train_test

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft_imputed.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]
print("PDX_COLS", len(PDX_COLS))

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()
print("pdXY", pdXY.shape)

std = Standardizer(exclude_cols=PDY_COLS)
std.fit(pdXY_train)

pdXY = std.transform(pdXY)
print(pdXY.shape)
display_df(pdXY.head())

pdXY.to_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft_imputed_std.csv", index=False)

## chembl33

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft_imputed.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]
print("PDX_COLS", len(PDX_COLS))

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()
print("pdXY", pdXY.shape)

std = Standardizer(exclude_cols=PDY_COLS)
std.fit(pdXY_train)


df00 = pd.read_csv("data/process/pdXY_chembl33_rdkit_descriptors_109ft_imputed.csv")
df00 = std.transform(df00)
print(df00.shape)
display_df(df00.head())

df00.to_csv("data/process/pdXY_chembl33_rdkit_descriptors_109ft_imputed_std.csv", index=False)

## chembl36excl33

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft_imputed.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]
print("PDX_COLS", len(PDX_COLS))

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()
print("pdXY", pdXY.shape)

std = Standardizer(exclude_cols=PDY_COLS)
std.fit(pdXY_train)


df00 = pd.read_csv("data/process/pdXY_chembl36excl33_rdkit_descriptors_109ft_imputed.csv")
df00 = std.transform(df00)
print(df00.shape)
display_df(df00.head())

df00.to_csv("data/process/pdXY_chembl36excl33_rdkit_descriptors_109ft_imputed_std.csv", index=False)

## amylovis201

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft_imputed.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]
print("PDX_COLS", len(PDX_COLS))

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()
print("pdXY", pdXY.shape)

std = Standardizer(exclude_cols=PDY_COLS)
std.fit(pdXY_train)


df00 = pd.read_csv("data/process/pdXY_amylovis201_rdkit_descriptors_109ft_imputed.csv")
df00 = std.transform(df00)
print(df00.shape)
display_df(df00.head())

df00.to_csv("data/process/pdXY_amylovis201_rdkit_descriptors_109ft_imputed_std.csv", index=False)

## ref01

In [ ]:
assert False

pdXY = pd.read_csv("data/process/pdXY_ab42_train_test_rdkit_descriptors_109ft_imputed.csv")
print(pdXY.shape)
display_df(pdXY.head())

PDY_COLS = ["new_id", "smiles", "dG", "code", "train_test", "smiles_len"]
PDX_COLS = [col for col in pdXY.columns if col not in PDY_COLS]
print("PDX_COLS", len(PDX_COLS))

pdXY_train = pdXY[pdXY["train_test"] == "train"].copy()
print("pdXY", pdXY.shape)

std = Standardizer(exclude_cols=PDY_COLS)
std.fit(pdXY_train)


df00 = pd.read_csv("data/process/pdXY_ref01_rdkit_descriptors_109ft_imputed.csv")
df00 = std.transform(df00)
print(df00.shape)
display_df(df00.head())

df00.to_csv("data/process/pdXY_ref01_rdkit_descriptors_109ft_imputed_std.csv", index=False)